In [1]:
%pip install transformers deep-translator rouge-score bert-score nltk pillow -q

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\HK6\DeepLearning\VQACK\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
# CELL 2: IMPORT
# ============================================================
import os
import torch
import pandas as pd
from PIL import Image
from transformers import BlipProcessor, BlipForQuestionAnswering
from deep_translator import GoogleTranslator

In [2]:
# CELL 3: LOAD DATASET
# ============================================================
train_df = pd.read_csv("Dataset/train.csv")
val_df   = pd.read_csv("Dataset/val.csv")
test_df  = pd.read_csv("Dataset/test.csv")
 
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
 
# Xem mẫu dữ liệu — xác nhận tiếng Việt
print("\nMẫu dữ liệu:")
print(test_df[["image_path", "question", "answer"]].head(3).to_string())

Train: 3960
Val  : 2376
Test : 1683

Mẫu dữ liệu:
                       image_path                         question                               answer
0  Dataset/Test/Banh canh/110.jpg                   Đây là món gì?                Đây là món bánh canh.
1  Dataset/Test/Banh canh/110.jpg      Tên món ăn trong ảnh là gì?             Tên món ăn là bánh canh.
2  Dataset/Test/Banh canh/110.jpg  Đây có phải món gỏi cuốn không?  Không, đây không phải món gỏi cuốn.


In [3]:
# CELL 4: CHIẾN LƯỢC XỬ LÝ TIẾNG VIỆT
# ============================================================
 
def translate_vi_to_en(text):
    try:
        return GoogleTranslator(source="vi", target="en").translate(text)
    except Exception as e:
        print(f"  [Translate VI→EN lỗi]: {e}")
        return text  # fallback: giữ nguyên
 
def translate_en_to_vi(text):
    try:
        return GoogleTranslator(source="en", target="vi").translate(text)
    except Exception as e:
        print(f"  [Translate EN→VI lỗi]: {e}")
        return text  # fallback: giữ nguyên
 
# Test thử
q_vi = "Đây là món gì?"
q_en = translate_vi_to_en(q_vi)
print(f"VI: {q_vi}")
print(f"EN: {q_en}")

VI: Đây là món gì?
EN: What is this?


In [4]:
# CELL 5: LOAD BLIP MODEL
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
 
processor_b1 = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
model_b1     = BlipForQuestionAnswering.from_pretrained(
    "Salesforce/blip-vqa-base",
    torch_dtype=torch.float32
)
model_b1.to(device)
model_b1.eval()
 
print("BLIP B1 loaded")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Device: cpu


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

BLIP B1 loaded


In [6]:
# CELL 6: HÀM GENERATE ANSWER B1
# ============================================================
def generate_answer_b1(image_path, question_vi):
    """
    Nhận vào: đường dẫn ảnh + câu hỏi tiếng Việt
    Trả về  : câu trả lời tiếng Việt
    Chiến lược: VI→EN → BLIP → EN→VI
    """
    question_en = translate_vi_to_en(question_vi)
 
    image = Image.open(image_path).convert("RGB")
 
    inputs = processor_b1(
        images=image,
        text=question_en,
        return_tensors="pt"
    ).to(device)
 
    with torch.no_grad():
        output_ids = model_b1.generate(
            **inputs,
            max_new_tokens=20,
            num_beams=5
        )
 
    answer_en = processor_b1.decode(output_ids[0], skip_special_tokens=True)
    answer_en = answer_en.lower().strip()
 
    answer_vi = translate_en_to_vi(answer_en)
 
    return answer_vi.lower().strip(), answer_en  # trả về cả 2 để debug
 

In [7]:
# CELL 7: TEST THỬ 1 ẢNH
# ============================================================
# Lấy 1 sample từ test set để kiểm tra
sample = test_df.iloc[0]
 
img_path    = sample["image_path"]
question_vi = sample["question"]
answer_ref  = sample["answer"]
 
answer_vi, answer_en = generate_answer_b1(img_path, question_vi)
 
print("===== TEST =====")
print(f"Image    : {img_path}")
print(f"Q (VI)   : {question_vi}")
print(f"Q (EN)   : {translate_vi_to_en(question_vi)}")
print(f"A (EN)   : {answer_en}")
print(f"A (VI)   : {answer_vi}")
print(f"Ref (VI) : {answer_ref}")

===== TEST =====
Image    : Dataset/Test/Banh canh/110.jpg
Q (VI)   : Đây là món gì?
Q (EN)   : What is this?
A (EN)   : soup
A (VI)   : canh
Ref (VI) : Đây là món bánh canh.


In [ ]:
# CELL 8: MULTI TEST — xem tổng quan chất lượng
# ============================================================
samples = test_df.head(5)

print("===== MULTI TEST B1 =====")
for _, row in samples.iterrows():
    ans_vi, ans_en = generate_answer_b1(row["image_path"], row["question"])
    print("\n" + "-"*40)
    print(f"Image  : {row['image_path']}")
    print(f"Q (VI) : {row['question']}")
    print(f"A (EN) : {ans_en}")
    print(f"A (VI) : {ans_vi}")
    print(f"Ref    : {row['answer']}")

===== MULTI TEST B1 =====

----------------------------------------
Image  : Dataset/Test/Banh canh/110.jpg
Q (VI) : Đây là món gì?
A (EN) : soup
A (VI) : canh
Ref    : Đây là món bánh canh.

----------------------------------------
Image  : Dataset/Test/Banh canh/110.jpg
Q (VI) : Tên món ăn trong ảnh là gì?
A (EN) : soup
A (VI) : canh
Ref    : Tên món ăn là bánh canh.

----------------------------------------
Image  : Dataset/Test/Banh canh/110.jpg
Q (VI) : Đây có phải món gỏi cuốn không?
A (EN) : no
A (VI) : không
Ref    : Không, đây không phải món gỏi cuốn.

----------------------------------------
Image  : Dataset/Test/Banh canh/110.jpg
Q (VI) : Đây thuộc loại món gì?
A (EN) : soup
A (VI) : canh
Ref    : Đây là món nước.

----------------------------------------
Image  : Dataset/Test/Banh canh/110.jpg
Q (VI) : Đây có phải món nước không?
A (EN) : no
A (VI) : không
Ref    : Đúng, đây là món nước.


In [ ]:
# CELL 9: EVALUATION PIPELINE B1
# ============================================================
import nltk
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_lib
from bert_score import score as bert_score_fn

smoother     = SmoothingFunction().method1
scorer_rouge = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)


def evaluate_b1(test_df):
    exact_matches = []
    bleu_scores   = []
    rougeL_scores = []
    preds_all     = []
    refs_all      = []

    print(f"\n🔍 Đánh giá B1 (Zero-shot BLIP) trên {len(test_df)} mẫu test...")
    print("   Chiến lược: câu hỏi VI→EN → BLIP → EN→VI\n")

    for i, (_, row) in enumerate(test_df.iterrows()):
        pred_vi, _ = generate_answer_b1(row["image_path"], row["question"])
        ref_vi     = row["answer"].lower().strip()
        pred_vi    = pred_vi.lower().strip()

        preds_all.append(pred_vi)
        refs_all.append(ref_vi)

        # Exact match
        exact_matches.append(int(pred_vi == ref_vi))

        # BLEU
        bleu = sentence_bleu(
            [ref_vi.split()],
            pred_vi.split(),
            smoothing_function=smoother
        )
        bleu_scores.append(bleu)

        # ROUGE-L
        rouge = scorer_rouge.score(ref_vi, pred_vi)
        rougeL_scores.append(rouge["rougeL"].fmeasure)

        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(test_df)}] done...")

    # BERTScore
    print("\n  Đang tính BERTScore...")
    P, R, F1 = bert_score_fn(preds_all, refs_all, lang="vi", verbose=False)

    results = {
        "model":        "B1 (BLIP zero-shot)",
        "exact_match":  sum(exact_matches) / len(test_df),
        "bleu":         sum(bleu_scores)   / len(test_df),
        "rougeL":       sum(rougeL_scores) / len(test_df),
        "bertscore_f1": F1.mean().item(),
        "preds":        preds_all,
        "refs":         refs_all,
    }

    print(f"\n  Exact Match : {results['exact_match']:.2%}")
    print(f"  BLEU        : {results['bleu']:.4f}")
    print(f"  ROUGE-L     : {results['rougeL']:.4f}")
    print(f"  BERTScore F1: {results['bertscore_f1']:.4f}")

    return results

In [8]:
# CELL 10: CHẠY ĐÁNH GIÁ B1
# ============================================================
results_b1 = evaluate_b1(test_df)


# ============================================================
# CELL 11: BIỂU ĐỒ KẾT QUẢ B1
# ============================================================
import matplotlib.pyplot as plt

metrics = ["exact_match", "bleu", "rougeL", "bertscore_f1"]
labels  = ["Exact Match", "BLEU", "ROUGE-L", "BERTScore F1"]
values  = [results_b1[m] for m in metrics]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("B1 — BLIP Zero-shot (VI→EN→BLIP→EN→VI)", fontsize=12)

for ax, label, val in zip(axes, labels, values):
    bar = ax.bar([label], [val], color="#55A868", width=0.4)
    ax.set_ylim(0, max(val * 1.4, 0.1))
    ax.set_title(label, fontsize=11)
    ax.text(0, val + 0.005, f"{val:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig("eval_b1.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Đã lưu: eval_b1.png")


NameError: name 'evaluate_b1' is not defined

In [11]:
# CELL 12: DEMO INTERACTIVE B1 
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io
import matplotlib.pyplot as plt
 
upload_b1   = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description=" Chọn ảnh",
    layout=widgets.Layout(width="200px")
)
 
question_b1 = widgets.Text(
    value="Đây là món gì?",
    description="Câu hỏi:",
    placeholder="Nhập câu hỏi tiếng Việt...",
    layout=widgets.Layout(width="500px")
)
 
button_b1  = widgets.Button(
    description=" Generate Answer",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="35px")
)
 
clear_btn  = widgets.Button(
    description=" Xóa",
    button_style="warning",
    layout=widgets.Layout(width="80px", height="35px")
)
 
output_b1  = widgets.Output()
 
preview_out = widgets.Output()
 
def on_upload_change(change):
    with preview_out:
        clear_output()
        if len(upload_b1.value) > 0:
            file  = upload_b1.value[0]
            image = Image.open(io.BytesIO(file["content"])).convert("RGB")
 
            # Resize preview cho gọn
            image_preview = image.copy()
            image_preview.thumbnail((300, 300))
 
            plt.figure(figsize=(3, 3))
            plt.imshow(image_preview)
            plt.axis("off")
            plt.title("Ảnh đã chọn", fontsize=10)
            plt.tight_layout()
            plt.show()
 
upload_b1.observe(on_upload_change, names="value")
 
def on_click_b1(b):
    with output_b1:
        clear_output()
 
        if len(upload_b1.value) == 0:
            print(" Hãy chọn ảnh trước")
            return
 
        if not question_b1.value.strip():
            print(" Hãy nhập câu hỏi")
            return
 
        print(" Đang xử lý...")
 
        file      = upload_b1.value[0]
        img_bytes = file["content"]
        image     = Image.open(io.BytesIO(img_bytes)).convert("RGB")
 
        temp_path = "temp_b1.jpg"
        image.save(temp_path)
 
        q_vi           = question_b1.value.strip()
        q_en           = translate_vi_to_en(q_vi)
        ans_vi, ans_en = generate_answer_b1(temp_path, q_vi)
 
        clear_output()
 
        print("=" * 45)
        print("  KẾT QUẢ B1 — BLIP Zero-shot")
        print("=" * 45)
        print(f"  Chiến lược : VI → EN → BLIP → EN → VI")
        print(f"  Q (VI)     : {q_vi}")
        print(f"  Q (EN)     : {q_en}")
        print(f"  A (EN)     : {ans_en}")
        print(f"  A (VI)     :  {ans_vi}")
        print("=" * 45)
 
def on_clear(b):
    with output_b1:
        clear_output()
    with preview_out:
        clear_output()
 
button_b1.on_click(on_click_b1)
clear_btn.on_click(on_clear)
 
title    = widgets.HTML("<h3 style='margin:0'>🖼️ Demo VQA — B1 Zero-shot BLIP</h3>")
subtitle = widgets.HTML("<p style='color:gray;margin:0'>Upload ảnh từ máy tính, nhập câu hỏi tiếng Việt → nhận câu trả lời</p>")
 
btn_row  = widgets.HBox([button_b1, clear_btn],
                        layout=widgets.Layout(gap="10px"))
 
left_col  = widgets.VBox(
    [upload_b1, preview_out],
    layout=widgets.Layout(width="320px", padding="8px")
)
 
right_col = widgets.VBox(
    [question_b1, btn_row, output_b1],
    layout=widgets.Layout(width="560px", padding="8px")
)
 
main_box = widgets.HBox(
    [left_col, right_col],
    layout=widgets.Layout(border="1px solid #ddd", border_radius="8px", padding="10px")
)
 
display(title, subtitle, main_box)

#có gì trong bức ảnh
#màu chính trong bức ảnh là gì 
#hành động trong bức ảnh là gì 


HTML(value="<h3 style='margin:0'>🖼️ Demo VQA — B1 Zero-shot BLIP</h3>")

HTML(value="<p style='color:gray;margin:0'>Upload ảnh từ máy tính, nhập câu hỏi tiếng Việt → nhận câu trả lời<…

In [10]:
# CELL 13: CÀI THÊM THƯ VIỆN CHO B2
# ============================================================
%pip install accelerate -q
 

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\HK6\DeepLearning\VQACK\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [14]:
# CELL 14: DỊCH DATASET VI→EN — LƯU CACHE (chỉ chạy 1 lần)
# ============================================================

 
import time
 
def translate_batch(texts, src="vi", tgt="en", delay=0.3):
    """Dịch danh sách texts, có retry 3 lần và delay tránh rate limit."""
    results = []
    for i, text in enumerate(texts):
        for attempt in range(3):
            try:
                translated = GoogleTranslator(source=src, target=tgt).translate(str(text))
                results.append(translated if translated else str(text))
                break
            except Exception as e:
                if attempt == 2:
                    print(f"  [Lỗi dịch idx={i}]: {e} → giữ nguyên")
                    results.append(str(text))
                else:
                    time.sleep(1)
        time.sleep(delay)
        if (i + 1) % 100 == 0:
            print(f"  Đã dịch {i+1}/{len(texts)}...")
    return results
 
 
def build_translated_cache(df, cache_path, split_name):
    """Load cache nếu có, nếu không thì dịch và lưu."""
    if os.path.exists(cache_path):
        print(f" [{split_name}] Cache tồn tại → load: {cache_path}")
        return pd.read_csv(cache_path)
 
    print(f"🔄 [{split_name}] Đang dịch {len(df)} mẫu...")
    df_en = df.copy()
 
    print(f"  Dịch question...")
    df_en["question_en"] = translate_batch(df["question"].tolist())
 
    print(f"  Dịch answer...")
    df_en["answer_en"] = translate_batch(df["answer"].tolist())
 
    df_en.to_csv(cache_path, index=False, encoding="utf-8")
    print(f" [{split_name}] Đã lưu: {cache_path}")
    return df_en
 
 
# Chạy dịch 
train_en = build_translated_cache(train_df, "Dataset/train_en.csv", "train")
val_en   = build_translated_cache(val_df,   "Dataset/val_en.csv",   "val")
test_en  = build_translated_cache(test_df,  "Dataset/test_en.csv",  "test")
 
print("\n Dataset tiếng Anh sẵn sàng")
print("\nMẫu sau khi dịch:")
print(train_en[["question", "question_en", "answer", "answer_en"]].head(3).to_string())
 

 [train] Cache tồn tại → load: Dataset/train_en.csv
 [val] Cache tồn tại → load: Dataset/val_en.csv
 [test] Cache tồn tại → load: Dataset/test_en.csv

 Dataset tiếng Anh sẵn sàng

Mẫu sau khi dịch:
                            question                                 question_en                                 answer                              answer_en
0                     Đây là món gì?                               What is this?                  Đây là món bánh canh.                   This is noodle soup.
1        Tên món ăn trong ảnh là gì?  What is the name of the dish in the photo?               Tên món ăn là bánh canh.     The name of the dish is Banh Canh.
2  Đây có phải món bún bò Huế không?               Is this Hue beef noodle soup?  Không, đây không phải món bún bò Huế.  No, this is not Hue beef noodle soup.


In [15]:
# CELL 15 (FIXED): BLIP DATASET
# ============================================================
from torch.utils.data import Dataset, DataLoader
 
 
class BLIPDataset(Dataset):
    def __init__(self, df_en, processor, max_len=32):
        self.df        = df_en.reset_index(drop=True)
        self.processor = processor
        self.max_len   = max_len
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
 
        image    = Image.open(row["image_path"]).convert("RGB")
        question = str(row["question_en"]).encode("ascii", errors="ignore").decode().strip()
        answer   = str(row["answer_en"]).encode("ascii", errors="ignore").decode().strip()
 
        if not question:
            question = "what is this"
        if not answer:
            answer = "unknown"
 
        # Encode image + question
        encoding = self.processor(
            images=image,
            text=question,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_len
        )
 
        # Encode answer → labels
        label_encoding = self.processor.tokenizer(
            answer,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_len
        )
 
        labels = label_encoding.input_ids
        labels = labels.masked_fill(
            labels == self.processor.tokenizer.pad_token_id, -100
        )
 
        return {
            "pixel_values":   encoding["pixel_values"].squeeze(0),
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         labels.squeeze(0),
        }
 
 
print(" BLIPDataset defined")

 BLIPDataset defined


In [17]:
# CELL 16 
# ============================================================
from transformers import BlipForConditionalGeneration
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
 
processor_b2 = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model_b2     = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float32
)
 
# Freeze TOÀN BỘ
for param in model_b2.parameters():
    param.requires_grad = False
 
# Chỉ unfreeze layer cuối cùng của text_decoder (nhẹ nhất)
# Layer cuối = layer[-1] của bert encoder
last_layer = model_b2.text_decoder.bert.encoder.layer[-1]
for param in last_layer.parameters():
    param.requires_grad = True
 
# Unfreeze output projection
for param in model_b2.text_decoder.cls.parameters():
    param.requires_grad = True
 
total     = sum(p.numel() for p in model_b2.parameters())
trainable = sum(p.numel() for p in model_b2.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
 
model_b2.to(device)
print(" BLIP B2 ready (minimal trainable)")
 

Device: cpu


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Trainable: 33,516,860 / 223,971,644 (15.0%)
 BLIP B2 ready (minimal trainable)


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [18]:
# CELL 17: DATALOADER 
# ============================================================
from torch.utils.data import Dataset, DataLoader
import gc
 
train_dataset_b2 = BLIPDataset(train_en, processor_b2)
val_dataset_b2   = BLIPDataset(val_en,   processor_b2)
 
train_loader_b2 = DataLoader(
    train_dataset_b2,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)
val_loader_b2 = DataLoader(
    val_dataset_b2,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)
 
# Test forward
print(" Test forward pass...")
sample_batch = next(iter(train_loader_b2))
test_batch   = {k: v.to(device) for k, v in sample_batch.items()}
with torch.no_grad():
    out = model_b2(**test_batch)
print(f"Forward pass OK — loss: {out.loss.item():.4f}")
del test_batch, out
gc.collect()

 Test forward pass...
Forward pass OK — loss: 6.7131


14595

In [16]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: False
GPU name: No GPU


In [8]:
# CELL DEBUG FINAL: Kiểm tra toàn bộ embedding layers trong BLIP

print("=== TẤT CẢ EMBEDDING LAYERS TRONG MODEL_B2 ===\n")

for name, module in model_b2.named_modules():
    if hasattr(module, 'weight') and 'embedding' in name.lower() and 'word' in name.lower():
        print(f"  {name}: {module.weight.shape}")

print("\n=== KIỂM TRA LABELS MAX ===")
# Lấy 1 batch và in ra labels thực tế
sample = next(iter(train_loader_b2))
labels = sample["labels"]
valid  = labels[labels != -100]
print(f"  labels max token id: {valid.max().item()}")
print(f"  labels min token id: {valid.min().item()}")

# Tìm xem token id nào > 30523
bad = valid[valid >= 30524]
print(f"  Số tokens vượt 30524: {len(bad)}")
if len(bad) > 0:
    print(f"  Token ids lỗi: {bad.unique().tolist()}")

print("\n=== KIỂM TRA INPUT_IDS MAX ===")
input_ids = sample["input_ids"]
print(f"  input_ids max: {input_ids.max().item()}")
print(f"  input_ids min: {input_ids.min().item()}")
bad_inp = input_ids[input_ids >= 30524]
print(f"  Số input_ids vượt 30524: {len(bad_inp)}")

print("\n=== TOKENIZER INFO ===")
print(f"  vocab_size          : {processor_b2.tokenizer.vocab_size}")
print(f"  len(tokenizer)      : {len(processor_b2.tokenizer)}")
print(f"  pad_token_id        : {processor_b2.tokenizer.pad_token_id}")
print(f"  sep_token_id        : {processor_b2.tokenizer.sep_token_id}")
print(f"  bos_token_id        : {processor_b2.tokenizer.bos_token_id}")
print(f"  eos_token_id        : {processor_b2.tokenizer.eos_token_id}")

=== TẤT CẢ EMBEDDING LAYERS TRONG MODEL_B2 ===

  text_encoder.embeddings.word_embeddings: torch.Size([30524, 768])
  text_decoder.bert.embeddings.word_embeddings: torch.Size([30524, 768])

=== KIỂM TRA LABELS MAX ===
  labels max token id: 29109
  labels min token id: 101
  Số tokens vượt 30524: 0

=== KIỂM TRA INPUT_IDS MAX ===
  input_ids max: 26156
  input_ids min: 0
  Số input_ids vượt 30524: 0

=== TOKENIZER INFO ===
  vocab_size          : 30522
  len(tokenizer)      : 30522
  pad_token_id        : 0
  sep_token_id        : 102
  bos_token_id        : None
  eos_token_id        : None


In [27]:
# CELL DEBUG: Kiểm tra vocab size thực tế
decoder_vocab = model_b2.text_decoder.bert.embeddings.word_embeddings.weight.shape[0]
tokenizer_vocab = processor_b2.tokenizer.vocab_size
tokenizer_total = len(processor_b2.tokenizer)  # bao gồm cả added tokens

print(f"Decoder embedding size  : {decoder_vocab}")
print(f"tokenizer.vocab_size    : {tokenizer_vocab}")
print(f"len(tokenizer)          : {tokenizer_total}")
print(f"→ Dùng giá trị         : {decoder_vocab}")

Decoder embedding size  : 30524
tokenizer.vocab_size    : 30522
len(tokenizer)          : 30522
→ Dùng giá trị         : 30524


In [ ]:
# CELL 18 (FIXED): TRAIN B2 — batch=1, accum=8, ít epoch
# ============================================================
from tqdm import tqdm
 
SAVE_PATH_B2    = "best_model_b2"
EPOCHS_B2       = 3             # CPU: 3 epoch đủ để fine-tune nhẹ
ACCUM_STEPS     = 8             # effective batch = 8
best_loss_b2    = float("inf")
train_losses_b2 = []
val_losses_b2   = []
 
optimizer_b2 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_b2.parameters()),
    lr=5e-5,
    weight_decay=0.01
)
 
total_steps  = (len(train_loader_b2) // ACCUM_STEPS) * EPOCHS_B2
scheduler_b2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_b2,
    T_max=max(total_steps, 1),
    eta_min=1e-6
)
 
if not os.path.exists(SAVE_PATH_B2):
    print("👉 Training B2...\n")
 
    for epoch in range(EPOCHS_B2):
 
        model_b2.train()
        total_loss = 0
        optimizer_b2.zero_grad()
 
        loop = tqdm(
            train_loader_b2,
            desc=f"[B2] Epoch {epoch+1}/{EPOCHS_B2}"
        )
 
        for step, batch in enumerate(loop):
            batch   = {k: v.to(device) for k, v in batch.items()}
            outputs = model_b2(**batch)
            loss    = outputs.loss / ACCUM_STEPS
 
            loss.backward()
            total_loss += loss.item() * ACCUM_STEPS
 
            if (step + 1) % ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model_b2.parameters()),
                    max_norm=1.0
                )
                optimizer_b2.step()
                scheduler_b2.step()
                optimizer_b2.zero_grad()
 
            loop.set_postfix(loss=f"{loss.item()*ACCUM_STEPS:.4f}")
 
            del batch, outputs, loss
            gc.collect()
 
        avg_train = total_loss / len(train_loader_b2)
        train_losses_b2.append(avg_train)
        print(f"[B2] Epoch {epoch+1} - Train Loss: {avg_train:.4f}")
 
        model_b2.eval()
        val_loss = 0
 
        with torch.no_grad():
            for batch in val_loader_b2:
                batch    = {k: v.to(device) for k, v in batch.items()}
                outputs  = model_b2(**batch)
                val_loss += outputs.loss.item()
                del batch, outputs
                gc.collect()
 
        avg_val = val_loss / len(val_loader_b2)
        val_losses_b2.append(avg_val)
        print(f"[B2] Epoch {epoch+1} - Val Loss  : {avg_val:.4f}")
 
        if avg_val < best_loss_b2:
            best_loss_b2 = avg_val
            model_b2.save_pretrained(SAVE_PATH_B2)
            processor_b2.save_pretrained(SAVE_PATH_B2)
            print(f"  Saved BEST B2 (val_loss={avg_val:.4f})")
 
    print("\n TRAIN B2 DONE")
 
else:
    print(" B2 model đã tồn tại, load lại...")
    model_b2 = BlipForConditionalGeneration.from_pretrained(
        SAVE_PATH_B2,
        torch_dtype=torch.float32
    )
    model_b2.to(device)
    model_b2.eval()
    print("Loaded B2 model")

👉 Training B2...



[B2] Epoch 1/3: 100%|██████████| 3960/3960 [52:23<00:00,  1.26it/s, loss=0.2291]


[B2] Epoch 1 - Train Loss: 1.3509
[B2] Epoch 1 - Val Loss  : 0.6039


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  🔥 Saved BEST B2 (val_loss=0.6039)


...

  🔥 Saved BEST B2 (val_loss=0.4967)


[B2] Epoch 3/3: 100%|██████████| 3960/3960 [55:34<00:00,  1.19it/s, loss=0.3161]


[B2] Epoch 3 - Train Loss: 0.4485
[B2] Epoch 3 - Val Loss  : 0.4735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  🔥 Saved BEST B2 (val_loss=0.4735)

✅ TRAIN B2 DONE


In [ ]:
# CELL 20 (FIXED V2): GENERATE ANSWER B2
# ============================================================
def generate_answer_b2(image_path, question_vi):
    model_b2.eval()

    # Bước 1: Dịch câu hỏi VI → EN
    question_en = translate_vi_to_en(question_vi)
    question_en = question_en.encode("ascii", errors="ignore").decode().strip()
    if not question_en:
        question_en = "what is this"

    image = Image.open(image_path).convert("RGB")

    # Bước 2: Encode inputs
    inputs = processor_b2(
        images=image,
        text=question_en,
        return_tensors="pt"
    ).to(device)

    # Bước 3: Generate
    with torch.no_grad():
        output_ids = model_b2.generate(
            **inputs,
            max_new_tokens=10,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=2,
            length_penalty=0.8
        )

    # Bước 4: Decode toàn bộ output
    full_text = processor_b2.tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    ).lower().strip()

    # Bước 5: Bỏ phần câu hỏi nếu model lặp lại
    # Tìm vị trí cuối câu hỏi trong output và lấy phần sau
    q_lower = question_en.lower().strip()
    if q_lower in full_text:
        answer_en = full_text.replace(q_lower, "").strip()
        # Xóa dấu ? hoặc . ở đầu nếu có
        answer_en = answer_en.lstrip("?.!, ").strip()
    else:
        answer_en = full_text

    if not answer_en:
        answer_en = full_text  # fallback

    # Bước 6: Dịch EN → VI
    answer_vi = translate_en_to_vi(answer_en)

    return answer_vi.lower().strip(), answer_en

In [29]:
# CELL 21: TEST THỬ B2

sample = test_df.iloc[0]

ans_vi_b2, ans_en_b2 = generate_answer_b2(sample["image_path"], sample["question"])

print("===== TEST THỬ B2 =====")
print(f"Image    : {sample['image_path']}")
print(f"Q  (VI)  : {sample['question']}")
print(f"Q  (EN)  : {translate_vi_to_en(sample['question'])}")
print(f"B2 (EN)  : {ans_en_b2}")
print(f"B2 (VI)  : {ans_vi_b2}")
print(f"Ref (VI) : {sample['answer']}")

ValueError: The following `model_kwargs` are not used by the model: ['decoder_input_ids'] (note: typos in the generate arguments will also show up in this list)

In [22]:
# CELL 22: MULTI TEST B2

samples = test_df.head(5)

print("===== MULTI TEST B2 =====")
for _, row in samples.iterrows():
    ans_vi_b2, ans_en_b2 = generate_answer_b2(row["image_path"], row["question"])

    print("\n" + "-"*50)
    print(f"Image    : {row['image_path']}")
    print(f"Q  (VI)  : {row['question']}")
    print(f"B2 (EN)  : {ans_en_b2}")
    print(f"B2 (VI)  : {ans_vi_b2}")
    print(f"Ref      : {row['answer']}")

===== MULTI TEST B2 =====

--------------------------------------------------
Image    : Dataset/Test/Banh canh/110.jpg
Q  (VI)  : Đây là món gì?
B2 (EN)  : what is this?????????????????????
B2 (VI)  : đây là cái gì thế?????????????????????
Ref      : Đây là món bánh canh.

--------------------------------------------------
Image    : Dataset/Test/Banh canh/110.jpg
Q  (VI)  : Tên món ăn trong ảnh là gì?
B2 (EN)  : what is the name of the dish in the photo??????????????????
B2 (VI)  : món ăn trong ảnh tên gì???????????????????
Ref      : Tên món ăn là bánh canh.

--------------------------------------------------
Image    : Dataset/Test/Banh canh/110.jpg
Q  (VI)  : Đây có phải món gỏi cuốn không?
B2 (EN)  : is this a spring roll dish? or is this a spring roll dish??
B2 (VI)  : đây có phải là món chả giò không? hay đây là món chả giò??
Ref      : Không, đây không phải món gỏi cuốn.

--------------------------------------------------
Image    : Dataset/Test/Banh canh/110.jpg
Q  (VI)  : Đâ

In [23]:
# CELL 23: EVALUATION PIPELINE B2
# ============================================================
import nltk
nltk.download("punkt", quiet=True)
 
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_lib
from bert_score import score as bert_score_fn
 
smoother     = SmoothingFunction().method1
scorer_rouge = rouge_lib.RougeScorer(["rougeL"], use_stemmer=False)
 
 
def evaluate_b2(test_df):
    exact_matches = []
    bleu_scores   = []
    rougeL_scores = []
    preds_all     = []
    refs_all      = []
 
    print(f"\n🔍 Đánh giá B2 (Fine-tuned BLIP) trên {len(test_df)} mẫu test...")
    print("   Chiến lược: Q(VI)→EN → BLIP fine-tuned → A(EN)→VI\n")
 
    for i, (_, row) in enumerate(test_df.iterrows()):
        pred_vi, _ = generate_answer_b2(row["image_path"], row["question"])
        ref_vi     = row["answer"].lower().strip()
        pred_vi    = pred_vi.lower().strip()
 
        preds_all.append(pred_vi)
        refs_all.append(ref_vi)
 
        exact_matches.append(int(pred_vi == ref_vi))
 
        bleu = sentence_bleu(
            [ref_vi.split()],
            pred_vi.split(),
            smoothing_function=smoother
        )
        bleu_scores.append(bleu)
 
        rouge = scorer_rouge.score(ref_vi, pred_vi)
        rougeL_scores.append(rouge["rougeL"].fmeasure)
 
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(test_df)}] done...")
 
    print("\n  Đang tính BERTScore...")
    P, R, F1 = bert_score_fn(preds_all, refs_all, lang="vi", verbose=False)
 
    results = {
        "model":        "B2 (BLIP fine-tuned)",
        "exact_match":  sum(exact_matches) / len(test_df),
        "bleu":         sum(bleu_scores)   / len(test_df),
        "rougeL":       sum(rougeL_scores) / len(test_df),
        "bertscore_f1": F1.mean().item(),
        "preds":        preds_all,
        "refs":         refs_all,
    }
 
    print(f"\n  ✅ Exact Match : {results['exact_match']:.2%}")
    print(f"  ✅ BLEU        : {results['bleu']:.4f}")
    print(f"  ✅ ROUGE-L     : {results['rougeL']:.4f}")
    print(f"  ✅ BERTScore F1: {results['bertscore_f1']:.4f}")
 
    return results

In [12]:
# CELL 24: CHẠY ĐÁNH GIÁ B2
# ============================================================
results_b2 = evaluate_b2(test_df)
 
 
# ============================================================
# CELL 25: SO SÁNH B1 vs B2
# ============================================================
import json
import matplotlib.pyplot as plt
 
# Load kết quả B1 đã lưu
with open("results_b1.json", "r", encoding="utf-8") as f:
    results_b1_loaded = json.load(f)
 
all_results = [results_b1_loaded, results_b2]
metrics     = ["exact_match", "bleu", "rougeL", "bertscore_f1"]
labels      = ["Exact Match", "BLEU", "ROUGE-L", "BERTScore F1"]
colors      = ["#4C72B0", "#DD8452"]
model_names = ["B1\n(Zero-shot)", "B2\n(Fine-tuned)"]
 
fig, axes = plt.subplots(1, 4, figsize=(14, 5))
fig.suptitle("B1 (Zero-shot) vs B2 (Fine-tuned BLIP) — Evaluation", fontsize=13)
 
for ax, metric, label in zip(axes, metrics, labels):
    vals = [results_b1_loaded[metric], results_b2[metric]]
    bars = ax.bar(model_names, vals, color=colors, width=0.45)
    ax.set_title(label, fontsize=11)
    ax.set_ylim(0, max(vals) * 1.4 + 0.01)
    for bar, v in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{v:.3f}", ha="center", va="bottom", fontsize=10
        )
 
plt.tight_layout()
plt.savefig("eval_b1_vs_b2.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Đã lưu: eval_b1_vs_b2.png")


🔍 Đánh giá B2 (Fine-tuned BLIP) trên 1683 mẫu test...
   Chiến lược: Q(VI)→EN → BLIP fine-tuned → A(EN)→VI

  [10/1683] done...
  [20/1683] done...
  [30/1683] done...
  [40/1683] done...
  [50/1683] done...
  [60/1683] done...
  [70/1683] done...
  [80/1683] done...
  [90/1683] done...
  [100/1683] done...
  [110/1683] done...
  [120/1683] done...
  [130/1683] done...
  [140/1683] done...
  [150/1683] done...
  [160/1683] done...
  [170/1683] done...
  [180/1683] done...
  [190/1683] done...
  [200/1683] done...
  [210/1683] done...
  [220/1683] done...
  [230/1683] done...
  [240/1683] done...
  [250/1683] done...
  [260/1683] done...
  [270/1683] done...
  [280/1683] done...
  [290/1683] done...
  [300/1683] done...
  [310/1683] done...
  [320/1683] done...
  [330/1683] done...
  [340/1683] done...
  [350/1683] done...
  [360/1683] done...
  [370/1683] done...
  [380/1683] done...
  [390/1683] done...
  [400/1683] done...
  [410/1683] done...
  [420/1683] done...
  [430/1683] done.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  ✅ Exact Match : 0.00%
  ✅ BLEU        : 0.1084
  ✅ ROUGE-L     : 0.4694
  ✅ BERTScore F1: 0.7428


FileNotFoundError: [Errno 2] No such file or directory: 'results_b1.json'

In [13]:
# CELL 26: LƯU KẾT QUẢ B2
# ============================================================
import json
 
summary_b2 = {k: v for k, v in results_b2.items() if k not in ["preds", "refs"]}
 
with open("results_b2.json", "w", encoding="utf-8") as f:
    json.dump(summary_b2, f, ensure_ascii=False, indent=2)
 
print(" Đã lưu: results_b2.json")
print(json.dumps(summary_b2, ensure_ascii=False, indent=2))
 

 Đã lưu: results_b2.json
{
  "model": "B2 (BLIP fine-tuned)",
  "exact_match": 0.0,
  "bleu": 0.10841539711673963,
  "rougeL": 0.46944284728520885,
  "bertscore_f1": 0.7428199052810669
}


In [24]:
# CELL 27: DEMO INTERACTIVE B2
# ============================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import io
import matplotlib.pyplot as plt
 
upload_b2   = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description=" Chọn ảnh",
    layout=widgets.Layout(width="200px")
)
 
question_b2 = widgets.Text(
    value="Đây là món gì?",
    description="Câu hỏi:",
    placeholder="Nhập câu hỏi tiếng Việt...",
    layout=widgets.Layout(width="500px")
)
 
button_b2  = widgets.Button(
    description=" Generate Answer",
    button_style="primary",
    layout=widgets.Layout(width="200px", height="35px")
)
 
clear_btn_b2 = widgets.Button(
    description=" Xóa",
    button_style="warning",
    layout=widgets.Layout(width="80px", height="35px")
)
 
output_b2   = widgets.Output()
preview_b2  = widgets.Output()

def on_upload_b2_change(change):
    with preview_b2:
        clear_output()
        if len(upload_b2.value) > 0:
            file  = upload_b2.value[0]
            image = Image.open(io.BytesIO(file["content"])).convert("RGB")
 
            image_preview = image.copy()
            image_preview.thumbnail((300, 300))
 
            plt.figure(figsize=(3, 3))
            plt.imshow(image_preview)
            plt.axis("off")
            plt.title("Ảnh đã chọn", fontsize=10)
            plt.tight_layout()
            plt.show()
 
upload_b2.observe(on_upload_b2_change, names="value")
 
def on_click_b2(b):
    with output_b2:
        clear_output()
 
        if len(upload_b2.value) == 0:
            print(" Hãy chọn ảnh trước")
            return
 
        if not question_b2.value.strip():
            print(" Hãy nhập câu hỏi")
            return
 
        print("⏳ Đang xử lý...")
 
        file      = upload_b2.value[0]
        img_bytes = file["content"]
        image     = Image.open(io.BytesIO(img_bytes)).convert("RGB")
 
        temp_path = "temp_b2.jpg"
        image.save(temp_path)
 
        q_vi           = question_b2.value.strip()
        q_en           = translate_vi_to_en(q_vi)
        ans_vi, ans_en = generate_answer_b2(temp_path, q_vi)
 
        clear_output()
 
        print("=" * 48)
        print("  KẾT QUẢ B2 — BLIP Fine-tuned")
        print("=" * 48)
        print(f"  Chiến lược : VI → EN → BLIP fine-tuned → EN → VI")
        print(f"  Q (VI)     : {q_vi}")
        print(f"  Q (EN)     : {q_en}")
        print(f"  A (EN)     : {ans_en}")
        print(f"  A (VI)     :  {ans_vi}")
        print("=" * 48)
 
def on_clear_b2(b):
    with output_b2:
        clear_output()
    with preview_b2:
        clear_output()
 
button_b2.on_click(on_click_b2)
clear_btn_b2.on_click(on_clear_b2)
 
title_b2    = widgets.HTML("<h3 style='margin:0'>Demo VQA — B2 Fine-tuned BLIP</h3>")
subtitle_b2 = widgets.HTML("<p style='color:gray;margin:0'>Upload ảnh từ máy tính, nhập câu hỏi tiếng Việt → nhận câu trả lời từ BLIP đã fine-tune</p>")
 
btn_row_b2  = widgets.HBox(
    [button_b2, clear_btn_b2],
    layout=widgets.Layout(gap="10px")
)
 
left_col_b2 = widgets.VBox(
    [upload_b2, preview_b2],
    layout=widgets.Layout(width="320px", padding="8px")
)
 
right_col_b2 = widgets.VBox(
    [question_b2, btn_row_b2, output_b2],
    layout=widgets.Layout(width="560px", padding="8px")
)
 
main_box_b2 = widgets.HBox(
    [left_col_b2, right_col_b2],
    layout=widgets.Layout(border="1px solid #ddd", border_radius="8px", padding="10px")
)
 
display(title_b2, subtitle_b2, main_box_b2)

#trong 

HTML(value="<h3 style='margin:0'>Demo VQA — B2 Fine-tuned BLIP</h3>")

HTML(value="<p style='color:gray;margin:0'>Upload ảnh từ máy tính, nhập câu hỏi tiếng Việt → nhận câu trả lời …